# A.3.2 ResNet 投影捷徑

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/03-projection/)

先讀兩路形狀、1×1 的通道混合與位置探針。這格先指定權重驗讀法，再另建隨機 block 檢查兩次 MSE 更新能改變投影權重。

先執行下一格的環境格，再執行最後的完整實驗。程式可獨立執行；概念請依 A 章閱讀順序學習。

保存的 CPU 輸出可和網頁核對，實驗不用預訓練權重或下載資料。


In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.1'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先讀兩路形狀、1×1 的通道混合與位置探針。這格先指定權重驗讀法，再另建隨機 block 檢查兩次 MSE 更新能改變投影權重。

先預測再執行，並對照網頁的停一下練習。


In [1]:
"""A projection aligns shapes and mixes channels; it is a learned transform."""
import torch
from torch import nn


class ProjectionBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = nn.Sequential(nn.Conv2d(3, 6, 3, stride=2, padding=1, bias=False),
                                    nn.ReLU(), nn.Conv2d(6, 6, 3, padding=1, bias=False))
        self.projection = nn.Conv2d(3, 6, 1, stride=2, bias=False)

    def forward(self, x):
        main, shortcut = self.branch(x), self.projection(x)
        assert main.shape == shortcut.shape
        return main + shortcut


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    block = ProjectionBlock()
    with torch.no_grad():
        for p in block.branch.parameters():
            p.zero_()
        block.projection.weight.zero_()
        block.projection.weight[0, :, 0, 0] = torch.tensor([1.0, 2.0, 3.0])
    image = torch.tensor([1.0, 10.0, 100.0]).view(1, 3, 1, 1).expand(1, 3, 4, 4)
    output = block(image)
    assert output.shape == (1, 6, 2, 2)
    assert torch.equal(output[0, 0], torch.full((2, 2), 321.0))
    print(f"input={tuple(image.shape)}, main={tuple(block.branch(image).shape)}, "
          f"projection={tuple(block.projection(image).shape)}, output={tuple(output.shape)}")
    print(f"first output channel={output[0, 0].detach().tolist()}; 1*1 + 2*10 + 3*100 = 321")
    # Position probe: the constant image cannot show which positions P reads. Here input
    # channel 0 holds 10*row + col and channels 1, 2 stay 0; its weight is 1, so each value
    # in the first output channel is the value P read (tens digit = row, units digit = col).
    probe = torch.zeros(1, 3, 4, 4)
    for row in range(4):
        for col in range(4):
            probe[0, 0, row, col] = 10 * row + col
    probe_output = block(probe)[0, 0].detach()
    assert torch.equal(probe_output, torch.tensor([[0.0, 2.0], [20.0, 22.0]]))  # rows/cols 0, 2
    print(f"position probe (input channel 0 = 10*row + col): "
          f"first output channel={probe_output.tolist()}")

    learned = ProjectionBlock()
    inputs = torch.randn(2, 3, 4, 4)
    target = torch.zeros(2, 6, 2, 2)
    optimizer = torch.optim.SGD(learned.parameters(), lr=0.05)
    before = learned.projection.weight.detach().clone()
    learned.train()
    for step in range(2):
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(learned(inputs), target)
        loss.backward()
        assert learned.projection.weight.grad.abs().sum() > 0
        optimizer.step()
        print(f"step={step}, loss={loss.item():.4f}")
    assert not torch.equal(before, learned.projection.weight)
    assert sum(p.numel() for p in learned.parameters()) == 504
    print("projection changed; block parameters=504; no classification-quality claim")


if __name__ == "__main__":
    main()


input=(1, 3, 4, 4), main=(1, 6, 2, 2), projection=(1, 6, 2, 2), output=(1, 6, 2, 2)
first output channel=[[321.0, 321.0], [321.0, 321.0]]; 1*1 + 2*10 + 3*100 = 321
position probe (input channel 0 = 10*row + col): first output channel=[[0.0, 2.0], [20.0, 22.0]]
step=0, loss=0.3553
step=1, loss=0.3350
projection changed; block parameters=504; no classification-quality claim
